<a href="https://colab.research.google.com/github/Luizadraeger/URBAN-DATA-ANALYTICS/blob/main/pipeline_main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Célula 3: Importa livrarias
import ee
import geemap
import geopandas as gpd
import os
import time
import pandas as pd

In [3]:
# Célula 4: Autenticação e inicialização dos serviços Google
# Esta célula configura o acesso ao Google Drive e ao Google Earth Engine (EE).

# PASSOS NECESSÁRIOS:
# 1. Monte o Google Drive para leitura/escrita de arquivos.
# 2. Autentique sua conta Google para uso do Earth Engine.
# 3. Inicialize o Earth Engine com um projeto válido do Google Cloud.

# IMPORTANTE:
# - Antes de executar, crie um projeto no Google Cloud Console.
# - Ative a API do Earth Engine nesse projeto.
# - Substitua 'thermal-luizadraeger' pelo ID do seu projeto.
# - Durante a execução, será solicitado um token de autenticação no Colab.

from google.colab import drive

# Monta o Google Drive no ambiente do Colab
# force_remount=True garante remontagem caso já esteja conectado
try:
    drive.mount('/content/drive', force_remount=True)
except:
    drive.mount('/content/drive')

# ─── Autenticação no Google Earth Engine ─────────────────────────────────────
# Abre um fluxo de login para autorizar o acesso ao EE
import ee
ee.Authenticate()

# Inicializa o Earth Engine com o projeto especificado
# Substitua pelo seu próprio ID de projeto no Google Cloud
ee.Initialize(project='thermal-luizadraeger')

Mounted at /content/drive


In [4]:
def buildfooprint(lat, lon, radius_m):
    """
    Extrai footprints e alturas de edifícios usando Google Open Buildings.
    """
    try:
        point = ee.Geometry.Point([lon, lat])
        region = point.buffer(radius_m).bounds()

        # Dataset de Alturas (Temporal V1)
        temporal_col = ee.ImageCollection('GOOGLE/Research/open-buildings-temporal/v1') \
                         .filterBounds(region) \
                         .sort('system:time_start', False)
        latest_raster = temporal_col.first()

        # Polígonos de Footprints (V3)
        buildings_fc = ee.FeatureCollection('GOOGLE/Research/open-buildings/v3/polygons') \
                         .filterBounds(region)

        # Amostragem de alturas por polígono
        sampled_fc = latest_raster.select('building_height').reduceRegions(
            collection=buildings_fc,
            reducer=ee.Reducer.median(),
            scale=1.0
        )

        features = sampled_fc.getInfo().get('features', [])
        building_list = []

        for f in features:
            props = f.get('properties', {})
            geom = f.get('geometry')
            if not geom: continue

            h_val = props.get('median')
            # Define altura padrão de 3.5m caso não encontre o dado
            height = float(h_val) if (h_val and h_val > 0) else 3.5

            # Extração de coordenadas para o Grasshopper
            coords = geom['coordinates'][0] if geom['type'] == 'Polygon' else geom['coordinates'][0][0]

            building_list.append({
                "geometry": coords,
                "height": height
            })

        return building_list
    except Exception as e:
        raise Exception(f"Erro no processamento GEE: {str(e)}")

In [8]:
def get_river_data(lat, lon, radius_m):
    """
    Extrai o trecho do rio paquequer a partir do shapefile local clonado do GitHub.
    """
    try:
        # 1. Definir o caminho local (após o !git clone no server)
        repo_path = '/content/URBAN-DATA-ANALYTICS/references/Hidrografia - Rio Paquequer'

        # Procurar arquivo .shp
        shp_file = next((f for f in os.listdir(repo_path) if f.endswith('.shp')), None)
        if not shp_file:
            return {"error": "Shapefile não encontrado no diretório."}

        # 2. Carregar o Shapefile
        full_path = os.path.join(repo_path, shp_file)
        gdf = gpd.read_file(full_path)

        # 3. Garantir CRS WGS84 (Lat/Lon)
        if gdf.crs != 'EPSG:4326':
            gdf = gdf.to_crs('EPSG:4326')

        # 4. Criar a geometria de recorte (Buffer em volta do ponto)
        # Importante: Como o buffer está em metros, usamos uma projeção métrica temporária para o recorte
        center_point = gpd.GeoSeries([Point(lon, lat)], crs='EPSG:4326')
        # Projeta para UTM (métrica) para fazer o buffer preciso em metros
        utm_crs = center_point.estimate_utm_crs()
        buffer_zone = center_point.to_crs(utm_crs).buffer(radius_m).to_crs('EPSG:4326').iloc[0]

        # 5. Realizar o recorte (Clip)
        river_clip = gdf[gdf.intersects(buffer_zone)].copy()

        # 6. Formatar para retorno (Lista de coordenadas para o Grasshopper)
        river_features = []
        for _, row in river_clip.iterrows():
            geom = row.geometry
            if geom.geom_type == 'LineString':
                coords = list(geom.coords)
            elif geom.geom_type == 'MultiLineString':
                coords = [list(line.coords) for line in geom.geoms]
            else:
                continue

            river_features.append({
                "type": geom.geom_type,
                "coordinates": coords
            })

        return river_features

    except Exception as e:
        return {"error": str(e)}